# Phase 1b: True Timestamp-Based Sensor Fusion (Research Grade)

In [ ]:
# Setup
%run colab_setup_updated.ipynb
from sklearn.ensemble import VotingClassifier
from collections import Counter
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
import json

print("✅ Research-grade setup complete!")


In [ ]:
# Filter to primary activities
wisdm_data = load_all_wisdm_data()
PRIMARY_ACTIVITIES = ['A', 'B', 'C', 'D', 'E']

def filter_primary(data):
    filtered = {}
    for key, df in data.items():
        if not df.empty:
            filtered[key] = df[df['activity'].isin(PRIMARY_ACTIVITIES)].copy()
    return filtered

primary_data = filter_primary(wisdm_data)


In [ ]:
# Override WINDOW_SIZE and STEP_SIZE
WINDOW_SIZE = 100  # 5 seconds
STEP_SIZE = 50     # 50% overlap

print(f"✅ Window size reduced to {WINDOW_SIZE} for faster mobile inference.")


In [ ]:
# -------------------------------------------------------------------------
# NEW LOGIC: Timestamp-based Fusion BEFORE feature extraction
# -------------------------------------------------------------------------

def align_raw_sensor_data(df_accel, df_gyro):
    '''Aligns raw accelerometer and gyroscope data using true timestamps.'''
    if df_accel is None or df_gyro is None or df_accel.empty or df_gyro.empty:
        return pd.DataFrame()
        
    print(f"   Accel shape: {df_accel.shape}, Gyro shape: {df_gyro.shape}")
    
    # Sort by timestamp
    df_accel = df_accel.sort_values('timestamp').dropna(subset=['timestamp'])
    df_gyro = df_gyro.sort_values('timestamp').dropna(subset=['timestamp'])
    
    # Prefix gyro columns to avoid collision
    df_gyro_renamed = df_gyro[['subject_id', 'activity', 'timestamp', 'x', 'y', 'z']].rename(
        columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'}
    )
    
    # Merge asof using timestamp, grouped by subject and activity
    merged = pd.merge_asof(
        df_accel, 
        df_gyro_renamed,
        on='timestamp',
        by=['subject_id', 'activity'],
        direction='nearest',
        tolerance=200000000  # 200ms tolerance
    )
    
    # Drop rows where merge failed (no nearby gyro reading)
    merged = merged.dropna(subset=['gyro_x', 'gyro_y', 'gyro_z'])
    print(f"   Merged shape: {merged.shape}")
    return merged

def process_fused_sensors(raw_merged, fusion_name):
    '''Extracts sliding windows and features from 6-axis merged data.'''
    if raw_merged.empty:
        return None, None, None
        
    windows = []
    labels = []
    subjects = []
    
    for subject_id in raw_merged['subject_id'].unique():
        subject_data = raw_merged[raw_merged['subject_id'] == subject_id]
        
        for activity in subject_data['activity'].unique():
            activity_data = subject_data[subject_data['activity'] == activity].sort_values('timestamp')
            
            xyz_accel = activity_data[['x', 'y', 'z']].values
            xyz_gyro = activity_data[['gyro_x', 'gyro_y', 'gyro_z']].values
            
            for start in range(0, len(activity_data) - WINDOW_SIZE, STEP_SIZE):
                window_accel = xyz_accel[start:start + WINDOW_SIZE]
                window_gyro = xyz_gyro[start:start + WINDOW_SIZE]
                windows.append((window_accel, window_gyro))
                labels.append(activity)
                subjects.append(subject_id)
                
    if len(windows) == 0:
        return None, None, None
        
    # Extract features for accel and gyro
    features = []
    for w_accel, w_gyro in tqdm(windows, desc=f"Extracting features for {fusion_name}"):
        # Extract accel features
        feat_accel = extract_features(np.array([w_accel])).iloc[0].to_dict()
        feat_accel = {f"{fusion_name}_accel_{k}": v for k, v in feat_accel.items()}
        
        # Extract gyro features
        feat_gyro = extract_features(np.array([w_gyro])).iloc[0].to_dict()
        feat_gyro = {f"{fusion_name}_gyro_{k}": v for k, v in feat_gyro.items()}
        
        # Merge dicts
        merged_feats = {**feat_accel, **feat_gyro}
        features.append(merged_feats)
        
    df_features = pd.DataFrame(features).replace([np.inf, -np.inf], np.nan).fillna(0)
    return df_features, np.array(labels), np.array(subjects)


In [ ]:
# Execute Timestamp Alignment
phone_merged = align_raw_sensor_data(primary_data.get('phone_accel'), primary_data.get('phone_gyro'))
X_phone_df, y_phone, subjects_phone = process_fused_sensors(phone_merged, 'phone')

watch_merged = align_raw_sensor_data(primary_data.get('watch_accel'), primary_data.get('watch_gyro'))
X_watch_df, y_watch, subjects_watch = process_fused_sensors(watch_merged, 'watch')


In [ ]:
# -------------------------------------------------------------------------
# NEW LOGIC: Strict Train/Test split preventing leakage + GroupKFold Validation
# -------------------------------------------------------------------------

def train_research_fusion_model(X_df, y, subjects, fusion_name):
    print(f"\n{'='*60}\n🔗 RESEARCH FUSION: {fusion_name.upper()}\n{'='*60}")
    
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # 1. Single Strict Holdout for Final Evaluation
    unique_subjects = np.unique(subjects)
    train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)
    
    # Save splits to JSON to prevent leakage in TFLite step
    splits = {'train': train_subjects.tolist(), 'test': test_subjects.tolist()}
    with open(f'{fusion_name}_subject_splits.json', 'w') as f:
        json.dump(splits, f)
    print(f"✅ Saved strict subject splits to prevent data leakage!")
    
    train_mask = np.isin(subjects, train_subjects)
    test_mask = np.isin(subjects, test_subjects)
    
    X_train, X_test = X_df.values[train_mask], X_df.values[test_mask]
    y_train, y_test = y_encoded[train_mask], y_encoded[test_mask]
    groups_train = subjects[train_mask]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 2. Hyperparameter tuning + Class Weights with GroupKFold
    print(f"\n   Training models with GroupKFold & Class Weighting...")
    
    # Fast XGBoost with Class Weights
    xgb_model = xgb.XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.05,
        random_state=42, use_label_encoder=False,
        eval_metric='mlogloss', n_jobs=-1
    )
    
    # RandomForest with Class Weights
    rf_model = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1)
    
    models = {'RandomForest': rf_model, 'XGBoost': xgb_model}
    results = {}
    
    # 3. K-Fold Validation on the Train set to prove robustness
    gkf = GroupKFold(n_splits=5)
    for name, model in models.items():
        print(f"\n      Evaluating {name}...")
        fold_f1s = []
        for fold_train_idx, fold_val_idx in gkf.split(X_train_scaled, y_train, groups=groups_train):
            X_f_train, X_f_val = X_train_scaled[fold_train_idx], X_train_scaled[fold_val_idx]
            y_f_train, y_f_val = y_train[fold_train_idx], y_train[fold_val_idx]
            model.fit(X_f_train, y_f_train)
            fold_f1s.append(f1_score(y_f_val, model.predict(X_f_val), average='weighted'))
        
        print(f"         5-Fold Mean F1: {np.mean(fold_f1s):.4f} (std: {np.std(fold_f1s):.4f})")
        
        # Train on full train set
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results[name] = {'model': model, 'accuracy': acc, 'f1': f1, 'predictions': y_pred}
        print(f"         Test Acc={acc:.4f}, Test F1={f1:.4f}")
        
    best_name = max(results, key=lambda x: results[x]['f1'])
    best_model = results[best_name]['model']
    best_f1 = results[best_name]['f1']
    
    print(f"\n   🏆 Best: {best_name} (F1: {best_f1:.4f})")
    
    model_dir = f'har_fusion_{fusion_name}'
    save_model(best_model, model_dir, scaler, le)
    joblib.dump(list(X_df.columns), os.path.join(MODELS_PATH, model_dir, 'feature_names.joblib'))
    
    return results[best_name]


In [ ]:
# Train Fusion Models
if X_phone_df is not None:
    phone_fusion_result = train_research_fusion_model(X_phone_df, y_phone, subjects_phone, 'phone')
    
print("✅ Phase 1b Research Upgrade Complete!")
